# ITBS-VoGen — Colab Training

Google Colab (T4 GPU) で RVC 話者モデルを学習するためのノートブック。ローカル (Mac) では推論のみ行い、重い学習だけクラウドに逃がす運用。

本ノートブックは **複数データセットから複数モデルを一度に学習する**構成。`SPEAKER_CONFIGS` に学習対象を列挙すると、依存セットアップは1回だけ走り、学習・保存は各モデル分ループする。

## 事前準備

1. **Google Drive に学習データをアップロード**: `MyDrive/itbs-vogen/Train/` 以下に各データセットフォルダ（例: `set1-1/`, `set1-2/`, `set1-3/`）を配置する。
2. **ランタイム変更**: メニュー → ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ = **T4 GPU**。
3. 以下のセルを上から順に実行。

## 所要時間の目安

T4 で 1 モデルあたり 20〜40 分（29分データ / 200 epoch 前提）。3モデルなら **合計1.5〜2時間**。Colab 無料枠のセッションタイムアウトに注意。途中で切れた場合は `SPEAKER_CONFIGS` から完了済みを消して再開すれば中断点から進められる。


## 1. GPU 確認

In [ ]:
!nvidia-smi

## 2. Google Drive マウント

学習データ読み込みと成果物保存に使用。認証プロンプトが出るので承認する。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. 学習設定

学習対象のデータセットを `SPEAKER_CONFIGS` に列挙する。各エントリは `(モデル名, Drive上のデータディレクトリ名)`。

`DRIVE_ROOT` 配下の想定構造:
```
itbs-vogen/
├── Train/
│   ├── set1-1/*.wav
│   ├── set1-2/*.wav
│   └── set1-3/*.wav
└── models/            ← 各モデルの .pth / .index がここに吐かれる
```


In [ ]:
import os
from pathlib import Path

# === 学習対象 ===
# (モデル名, Drive 上のデータディレクトリ名)
SPEAKER_CONFIGS = [
    ('itbs_set1_1', 'set1-1'),
    ('itbs_set1_2', 'set1-2'),
    ('itbs_set1_3', 'set1-3'),
]

# === 共通ハイパーパラメータ ===
TOTAL_EPOCHS = 200                 # 典型値: 150-300
SAVE_EVERY_EPOCH = 50              # 途中保存の粒度
BATCH_SIZE = 12                    # T4 なら 12、A100 なら 24+
SR = '48k'                         # 32k / 40k / 48k
VERSION = 'v2'

# === Drive 上のパス ===
DRIVE_ROOT = Path('/content/drive/MyDrive/itbs-vogen')
DRIVE_TRAIN_ROOT = DRIVE_ROOT / 'Train'
DRIVE_MODELS_ROOT = DRIVE_ROOT / 'models'

# === Colab 作業ディレクトリ ===
WORK_DIR = Path('/content/ITBS-VoGen')

# 事前チェック
DRIVE_MODELS_ROOT.mkdir(parents=True, exist_ok=True)
for name, subdir in SPEAKER_CONFIGS:
    src = DRIVE_TRAIN_ROOT / subdir
    assert src.exists(), f'Training data not found: {src}'
    wavs = list(src.glob('*.wav'))
    print(f'[{name}] {src} -> {len(wavs)} wav files')


## 4. リポジトリ取得

公開リポジトリとRVC submoduleを取得。

In [ ]:
%cd /content
![ -d /content/ITBS-VoGen ] || git clone --recurse-submodules https://github.com/Arimuri/ITBS-VoGen.git
%cd /content/ITBS-VoGen
!git pull
!git submodule update --init --recursive

## 5. Python 3.10 + 依存インストール

Colab のデフォルト Python は 3.12 だが、RVC のピン (`numba==0.56.4`, `fairseq==0.12.2`) が 3.12 非対応なので **Python 3.10 を apt で追加**し、以降は `python3.10 -m pip` で明示的に使う。

5-10分かかる。途中で「Gemini が pyproject.toml を直しましょうか?」という提案が出ても **拒否**すること（こちらで設計済みの制約を知らないので、勝手に依存を足すと RVC と衝突する）。万一改変されたら `!git checkout pyproject.toml` で戻せる。


In [ ]:
%cd /content/ITBS-VoGen

# Defensive reset: if anything (Gemini, a failed edit, a stale state) modified
# our pinned project metadata, revert it before installing.
!git checkout pyproject.toml

# ---- Python 3.10 -----------------------------------------------------------
# Colab defaults have moved to 3.12; RVC pins (numba 0.56.4, fairseq 0.12.2) require 3.10.
# Try the distro package first; fall back to the deadsnakes PPA if not available.
!apt-get update -q
!apt-get install -y -q python3.10 python3.10-dev python3.10-venv python3.10-distutils \
  || ( apt-get install -y -q software-properties-common \
       && add-apt-repository -y ppa:deadsnakes/ppa \
       && apt-get update -q \
       && apt-get install -y -q python3.10 python3.10-dev python3.10-venv python3.10-distutils )
!python3.10 --version

# Install pip for the freshly-added Python 3.10.
!curl -sS https://bootstrap.pypa.io/get-pip.py -o /tmp/get-pip.py
!python3.10 /tmp/get-pip.py --quiet
!python3.10 -m pip --version

# ---- Dependencies ----------------------------------------------------------
# RVC ships an old omegaconf whose metadata trips pip >= 24.1.
!python3.10 -m pip install --quiet 'pip<24.1'

# RVC's deps + our package (all inside python3.10).
# torch version is not pinned; the fairseq/torch.load weights_only issue is
# handled at runtime by src/itbs_vogen/_compat/sitecustomize.py (injected
# via PYTHONPATH when the wrapper spawns RVC subprocesses).
!python3.10 -m pip install --quiet -r third_party/rvc/requirements.txt
!python3.10 -m pip install --quiet -e .

# ---- Verify ----------------------------------------------------------------
!python3.10 -c "import sys, torch, fairseq, faiss; \
  print('py:', sys.version.split()[0]); \
  print('torch:', torch.__version__, 'cuda:', torch.cuda.is_available()); \
  print('fairseq:', fairseq.__version__); print('faiss ok')"
!python3.10 -m itbs_vogen.cli --help


## 6. ベースモデル取得

HuBERT + RMVPE + pretrained_v2 (48k, F0あり) をDL。初回のみ。

In [ ]:
!bash scripts/download_models.sh


## 7. 学習データ配置

Drive の各データセットを Colab ローカル (`/content/ITBS-VoGen/Train/<dir>/`) にコピーする (Drive直読みは遅い)。


In [ ]:
import shutil

for _, subdir in SPEAKER_CONFIGS:
    src = DRIVE_TRAIN_ROOT / subdir
    dst = WORK_DIR / 'Train' / subdir
    dst.mkdir(parents=True, exist_ok=True)
    for wav in src.glob('*.wav'):
        target = dst / wav.name
        if not target.exists():
            shutil.copy(wav, target)
    print(f'[{subdir}] copied {len(list(dst.glob("*.wav")))} wavs to {dst}')


## 8. プリフライトチェック

学習開始前に全ての前提条件が揃っているか確認。赤字が出たら該当するセルに戻って再実行すること。


In [ ]:
import sys
from pathlib import Path

problems = []

# Pretrained base models (downloaded by scripts/download_models.sh).
required = [
    WORK_DIR / 'third_party/rvc/assets/hubert/hubert_base.pt',
    WORK_DIR / 'third_party/rvc/assets/rmvpe/rmvpe.pt',
    WORK_DIR / f'third_party/rvc/assets/pretrained_v2/f0G{SR}.pth',
    WORK_DIR / f'third_party/rvc/assets/pretrained_v2/f0D{SR}.pth',
]
for p in required:
    if not p.exists():
        problems.append(f'MISSING pretrained: {p}  (re-run "6. ベースモデル取得")')
    else:
        print(f'ok  pretrained {p.name}  ({p.stat().st_size/1e6:.1f}MB)')

# Training data must be copied into the container.
for name, subdir in SPEAKER_CONFIGS:
    local = WORK_DIR / 'Train' / subdir
    wavs = list(local.glob('*.wav')) if local.exists() else []
    if not wavs:
        problems.append(f'MISSING training data: {local}  (re-run "7. 学習データ配置")')
    else:
        print(f'ok  {name}: {len(wavs)} wavs in {local}')

# Python 3.10 + imports must all work.
import subprocess as sp
r = sp.run(['python3.10', '-c',
    'import torch; assert torch.cuda.is_available(), "CUDA not available!"'],
    capture_output=True, text=True)
if r.returncode != 0:
    problems.append(f'Python 3.10 CUDA check failed:\n{r.stdout}\n{r.stderr}')
else:
    print('ok  python3.10 + CUDA')

if problems:
    print('\n=== PROBLEMS FOUND ===')
    for p in problems:
        print('  - ' + p)
    raise RuntimeError(f'{len(problems)} precondition(s) failed. Fix and rerun this cell.')
print('\nAll preconditions met. Proceed to training.')


## 9. 学習実行 (ループ)

`SPEAKER_CONFIGS` の全エントリを順番に学習する。T4 で各モデル 20-40 分、合計で数時間。サブプロセスの出力（RVC の各ステージ: preprocess, f0, feature, train, index）がリアルタイムで流れる。

各モデル完成直後に Drive に保存されるので、途中でセッションが切れても完了済みは残る。


In [ ]:
import subprocess
import time

for name, subdir in SPEAKER_CONFIGS:
    local_train = WORK_DIR / 'Train' / subdir
    drive_out = DRIVE_MODELS_ROOT / name
    drive_out.mkdir(parents=True, exist_ok=True)

    if (drive_out / 'model.pth').exists() and (drive_out / 'model.index').exists():
        print(f'[{name}] already trained (found in Drive). Skipping.')
        continue

    print(f'\n===== training {name} from {local_train} =====')
    t0 = time.time()
    cmd = [
        'python3.10', '-u', '-m', 'itbs_vogen.cli', 'train',
        '-s', name,
        '-d', str(local_train),
        '--sr', SR,
        '--version', VERSION,
        '--epochs', str(TOTAL_EPOCHS),
        '--save-every', str(SAVE_EVERY_EPOCH),
        '--batch-size', str(BATCH_SIZE),
        '--n-proc', '4',
        '--device', 'cuda:0',
    ]
    # Stream output live and raise with context if the subprocess fails.
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end='')
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f'Training for {name} failed with exit {proc.returncode}. See output above.')

    local_model = WORK_DIR / 'models' / 'speakers' / name
    for fname in ('model.pth', 'model.index'):
        src = local_model / fname
        if src.exists():
            shutil.copy(src, drive_out / fname)
    elapsed = time.time() - t0
    print(f'[{name}] done in {elapsed/60:.1f} min, saved to {drive_out}')


## 10. 保存確認

全モデルが Drive に保存されているか確認。


In [ ]:
for name, _ in SPEAKER_CONFIGS:
    drive_out = DRIVE_MODELS_ROOT / name
    pth = drive_out / 'model.pth'
    idx = drive_out / 'model.index'
    pth_ok = pth.exists()
    idx_ok = idx.exists()
    pth_size = f'{pth.stat().st_size / 1e6:.1f}MB' if pth_ok else 'missing'
    idx_size = f'{idx.stat().st_size / 1e6:.1f}MB' if idx_ok else 'missing'
    print(f'[{name}] pth: {pth_size}  |  index: {idx_size}')


## 11. ローカル (Mac) で使う

Mac 側で以下を実行:

1. Google Drive から 各モデルの `model.pth` + `model.index` をダウンロード
2. ローカルリポジトリの `models/speakers/<モデル名>/` に配置:
   ```
   models/speakers/itbs_set1_1/model.pth
   models/speakers/itbs_set1_1/model.index
   models/speakers/itbs_set1_2/model.pth
   models/speakers/itbs_set1_2/model.index
   models/speakers/itbs_set1_3/model.pth
   models/speakers/itbs_set1_3/model.index
   ```
3. 推論（各モデルで同じ入力を処理して聴き比べる想定）:
   ```bash
   itbs-vogen infer inputs/test_HonestyVo.wav \
       -o outputs/test_HonestyVo_set1_1.wav -s itbs_set1_1
   itbs-vogen infer inputs/test_HonestyVo.wav \
       -o outputs/test_HonestyVo_set1_2.wav -s itbs_set1_2
   itbs-vogen infer inputs/test_HonestyVo.wav \
       -o outputs/test_HonestyVo_set1_3.wav -s itbs_set1_3
   ```

## トラブルシュート

- **OOM (CUDA out of memory)**: `BATCH_SIZE` を 8, 6, 4 と下げる。
- **Gemini が pyproject.toml を書き換えた**: セル 5 を再実行すれば先頭で `git checkout pyproject.toml` が走って元に戻る。
- **学習途中でセッション切断**: 完了済みモデルは Drive に保存されているので、再接続後に「9. 学習実行」を再実行すれば完了済みを自動スキップして残りから再開。
- **サブプロセスが exit 1 で落ちる**: 「9. 学習実行」のセルはリアルタイムで出力を出す設計。エラーログが見えるはずなので上にスクロールして該当箇所を確認。
- **品質が低い**: epoch数を増やす (300-500)、または学習データを追加して再学習。
